Imports and environment

In [1]:
from dotenv import load_dotenv
import requests
import os
from firecrawl import Firecrawl
from IPython.display import Markdown, display
import json
from datetime import datetime
import re
from urllib.parse import urlsplit
from bs4 import BeautifulSoup
import os
import time

load_dotenv("../.env")

api_key = os.getenv("FIRECRAWL_API_KEY")

print(api_key[:10])

fc-f48a711


Archive selection logic and slug extraction

In [ ]:
response = requests.get("https://www.morningbrew.com/api/sanity/issues/archive?end=3000")
print(response.status_code)
print(response.text[:500])

# Doing a simple request still doesn't work

In [ ]:
# Testing scraping with Firecrawl again

client = Firecrawl(api_key=api_key)

result = client.scrape(
    "https://www.morningbrew.com/api/sanity/issues/archive?end=3000",
    formats=["markdown"]
)

display(Markdown(result.markdown))

In [ ]:
display(result.markdown)

In [ ]:
# The relevant info is in a json code block so the characters need to be removed

result.markdown.strip("'`json\n")

In [ ]:
full_archive = json.loads(result.markdown.strip("'`json\n"))

In [8]:
# Testing datetime functionality

date = datetime.strptime((full_archive[0]["date"].split("T"))[0],"%Y-%m-%d").date()
date

datetime.date(2026, 8, 16)

In [6]:
cutoff = datetime.fromisoformat("2020-10-31").date()
cutoff

datetime.date(2020, 10, 31)

In [2]:
# Combining above cells to create function and store archive

def scrape_archive_metadata():
    client = Firecrawl(api_key=api_key)

    result = client.scrape(
        "https://www.morningbrew.com/api/sanity/issues/archive?end=3000",
        formats=["markdown"]
    )

    return json.loads(result.markdown.strip("'`json\n"))

full_archive_metadata = scrape_archive_metadata()


In [ ]:
# Define date cutoffs and valid weekdays

first_thursday = datetime.fromisoformat("2020-11-05").date()
first_worded_slug = datetime.fromisoformat("2021-09-30").date()
valid_weekdays = [3,5] # Thursday, Saturday

# Function to process the date item into a usable Python date

def processed_date(item):
    return datetime.strptime((item["date"].split("T"))[0],"%Y-%m-%d").date()

# Extraction function with logic

def extract_slug(item):
    date = processed_date(item)
    if date.weekday() not in valid_weekdays:
        return None
    elif date.weekday() == 3 and date >= first_thursday:
        if date >= first_worded_slug:
            return item["slug"]
        else:
            return item["slug"].replace("/", "%2F")
    elif date.weekday() == 5 and date < first_thursday:
        return item["slug"].replace("/", "%2F")


In [ ]:
# Testing the extraction function on an individual edition

print(extract_slug(full_archive_metadata[64]))

'its-just-field'

In [ ]:
# Looping over all editions and printing to check it is working as intended

slugs = []

for item in full_archive_metadata:
    if extract_slug(item):
        print(extract_slug(item))
        slugs.append(extract_slug(item))
        #print(processed_date(item).weekday())
    else:
        continue

print(len(slugs))

Extracting raw URLs from Morning Brew links

In [ ]:
response = requests.get("https://links.morningbrew.com/c/U6n?mblid=f92dc3fd9d90&mbnlid=slV5a3vuYndT7PDVh24Aj8")
print(response.text)

In [ ]:
# I tried Beautiful Soup first
soup = BeautifulSoup(response.text, 'html.parser')
soup.find('link', attrs={'rel': 'canonical'}).get('href')

In [ ]:
# But then realised that his works too (I've now changed it to .headers.get("Location") instead of .url as some websites hung without response)

response = requests.get("https://links.morningbrew.com/c/U6q?mblid=0d6a557bf158&mbnlid=slV5a3vuYndT7PDVh24Aj8")
print(response.url)
o = urlsplit(response.url)
o._replace(query="").geturl()

In [ ]:
# Testing an unresponsive URL and how to get the right destination

response = requests.get("https://links.morningbrew.com/c/Ox0?mblid=19b17c4e99da&mbnlid=J4PVUnjbNhxkECeFjuBu2S", allow_redirects=False, timeout=10)
print(response.status_code)
print(response.headers.get("Location"))
url = response.headers.get("Location")
urlsplit(url)._replace(query="").geturl()

302
https://www.cbc.ca/news/canada/toronto/iceman-release-date-drake-ice-pile-9.7171479?utm_campaign=mb&utm_medium=newsletter&utm_source=morning_brew


'https://www.cbc.ca/news/canada/toronto/iceman-release-date-drake-ice-pile-9.7171479'

Archiving

In [95]:
iso_date = date.isoformat()
print(f"iso date: {iso_date}")

# for edition in archive:
#     if edition["date"] == iso_date:
#         exists = True
#         exit
#     else:
#         exists = False

exists = any(edition["date"] == iso_date for edition in archive)
exists


iso date: 2025-01-16


True

# The full process

In [ ]:
# Constants and initial steps (just testing on one edition)

first_thursday = datetime.fromisoformat("2020-11-05").date()
first_worded_slug = datetime.fromisoformat("2021-09-30").date()
valid_weekdays = [3,5] # Thursday, Saturday
index = 86
edition_metadata = full_archive_metadata[index]

def processed_date(edition_metadata):
    return datetime.strptime((edition_metadata["date"].split("T"))[0],"%Y-%m-%d").date()

id = edition_metadata["_id"]
date = processed_date(edition_metadata)
edition_metadata

{'_id': '0rMtBzRcQOSiUlzbv9S3Ff',
 'date': '2026-05-21T06:00:00-04:00',
 'previewText': "Elon Musk's space company made its IPO plans public...",
 'slug': 'spacex-ipo-ready-for-blastoff',
 'subjectLine': 'SpaceX IPO ready for blastoff',
 'title': None,
 'websitePreviewText': None}

In [ ]:
data_path = "../data"

with open(f"{data_path}/archive.json", "r") as f:
    archive = json.load(f)

archive

In [106]:
archive = []
quizzes = []

In [ ]:
# Check to see if edition exits, scrape it if not and store in archive

data_path = "../data"

def edition_condition_and_store(date, index, full_archive_metadata):

    # Check to see if archive exists, if not create blank list
    if os.path.isfile(f"{data_path}/archive.json"):
        with open(f"{data_path}/archive.json", "r") as f:
            archive = json.load(f)
    else:
        archive = []
    if os.path.isfile(f"{data_path}/quizzes.json"):
            with open(f"{data_path}/quizzes.json", "r") as f:
                quizzes = json.load(f)
    else:
        quizzes = []

    iso_date = date.isoformat()

    if any(archive_edition["date"] == iso_date for archive_edition in archive):
        return f"Edition for {iso_date} found"
    else:
        process_edition(index, full_archive_metadata, date, archive, quizzes)

    with open(f"{data_path}/archive.json", "w") as f:
        json.dump(archive, f, indent=2)

    with open(f"{data_path}/quizzes.json", "w") as f:
        json.dump(quizzes, f, indent=2)
    
edition_condition_and_store(date, index, full_archive_metadata)

In [ ]:
quizzes

In [ ]:
# Refactoring the single edition processing into a function so I can add logic to determine whether it already exists

def process_edition(index, full_archive_metadata, date, archive, quizzes):
    edition_metadata = full_archive_metadata[index]
    id = edition_metadata["_id"]
    slug = extract_slug(edition_metadata, date)
    scraped_edition = scrape_edition(slug)
    print("Edition scraped")
    edition = {
        "_id" : id,
        "date" : date.isoformat(),
        "slug" : slug,
        "content" : scraped_edition.markdown
    }
    questions_and_answers = filter_parts(scraped_edition)
    questions_and_answers = clean_q4_and_answer(questions_and_answers)

    quiz = {
            "_id": id,
            "edition_date": date.isoformat(),
            "source_url": f"https://www.morningbrew.com/issues/{slug}",
            "headlines": [],
            "answer": questions_and_answers[4]
        }

    print("Extracting headline text and URLs")
    for index, item in enumerate(questions_and_answers):
        if item[0].isalpha():
            continue
        else:
            headline, url = url_headline_extract(index, questions_and_answers)
            quiz["headlines"].append({"text": headline, "url": url})

    print(json.dumps(quiz, indent=2))

    return archive.append(edition), quizzes.append(quiz)

    # time.sleep(6)

# process_edition(index, full_archive_metadata, date)


## Individual functions

In [68]:
# Selection logic for whether there will be a quiz to find in the edition

def selection_logic(date):
    if date.weekday() not in valid_weekdays:
        return False
    elif date.weekday() == 3 and date >= first_thursday:
        if date >= first_worded_slug:
            return True
        else:
            return True
    elif date.weekday() == 5 and date < first_thursday:
        return True
    else:
        return False

In [155]:
selection_logic(date)

True

In [70]:
# Extract the slug

def extract_slug(edition_metadata, date):
    if date.weekday() not in valid_weekdays:
        return None
    elif date.weekday() == 3 and date >= first_thursday:
        if date >= first_worded_slug:
            return edition_metadata["slug"]
        else:
            return edition_metadata["slug"].replace("/", "%2F")
    elif date.weekday() == 5 and date < first_thursday:
        return edition_metadata["slug"].replace("/", "%2F")

In [156]:
slug = extract_slug(edition_metadata, date)
print(slug)

spacex-ipo-ready-for-blastoff


In [72]:
# Scrape edition

def scrape_edition(slug):
    client = Firecrawl(api_key=api_key)
    scraped_edition = client.scrape(
        f"https://www.morningbrew.com/issues/{slug}",
        formats=["markdown"]
    )

    return scraped_edition



In [157]:
scraped_edition = scrape_edition(slug)
scraped_edition.markdown

'[Skip to main content](https://www.morningbrew.com/issues/spacex-ipo-ready-for-blastoff#main-content)\n\nWelcome! We use third-party tools to collect information about you as you enjoy the site. Our [Privacy Policy](https://privacy.morningbrewinc.com/policies?utm_medium=website&utm_source=terms_banner&utm_campaign=mb) explains more. To opt-out of targeted ads, choose [Do Not Sell / Share My Info](https://privacy.morningbrewinc.com/policies?utm_medium=website&utm_source=terms_banner&utm_campaign=mb). By proceeding, you accept our [Terms](https://www.morningbrew.com/terms-service?utm_medium=website&utm_source=terms_banner&utm_campaign=mb) and the [Policy](https://privacy.morningbrewinc.com/policies?utm_medium=website&utm_source=terms_banner&utm_campaign=mb).\n\nGot it\n\n###### ☕SpaceX IPO ready for blastoff\n\nTo:Brew Readers\n\n* * *\n\n|     |     |     |     |\n| --- | --- | --- | --- |\n| |     |\n| --- |\n| Elon Musk\'s space company made its IPO plans public... |\n\n[![](https://

In [152]:
scraped_edition

Document(markdown='[Skip to main content](https://www.morningbrew.com/issues/its-just-field#main-content)\n\nWelcome! We use third-party tools to collect information about you as you enjoy the site. Our [Privacy Policy](https://privacy.morningbrewinc.com/policies?utm_medium=website&utm_source=terms_banner&utm_campaign=mb) explains more. To opt-out of targeted ads, choose [Do Not Sell / Share My Info](https://privacy.morningbrewinc.com/policies?utm_medium=website&utm_source=terms_banner&utm_campaign=mb). By proceeding, you accept our [Terms](https://www.morningbrew.com/terms-service?utm_medium=website&utm_source=terms_banner&utm_campaign=mb) and the [Policy](https://privacy.morningbrewinc.com/policies?utm_medium=website&utm_source=terms_banner&utm_campaign=mb).\n\nGot it\n\n###### ☕It’s just Field\n\nTo:Brew Readers\n\n* * *\n\n|     |\n| --- |\n| Inflation reached 4.2% in May… |\n\n[![](https://media.sailthru.com/5z8/1k3/3/k/5c9296dc6e8c2.png)](http://www.morningbrew.com/)\n\n|     |  

In [121]:
# Cut everything before the quiz and break into parts

def cut_before_and_split(scraped_edition):
    # I decided to use a regex instead of the below as it's more robust
    # if date >= first_thursday:
    #     quiz_section_start_index = scraped_edition.markdown.lower().find("three headlines and") # Removed "a lie" as there was an edition with a typo missing out "a"
    # if date < first_thursday:
    #     quiz_section_start_index = scraped_edition.markdown.lower().find("saturday headlines")
    split_quiz = scraped_edition.markdown.split("<br>")
    # q1_index = re.compile(r'1\. (“)?.*\[([\w\s\.])*\](\(.*\))').search(scraped_edition.markdown.lower()).start()
    # quiz_section = scraped_edition.markdown[q1_index:]
    # return quiz_section.split("<br>")
    return split_quiz


In [143]:
q_search = re.compile(r'[1234]\. (“)?.*\[([\w\s\.])*\](\(.*\))')
matched = q_search.search(parts[72])
matched

<re.Match object; span=(0, 121), match='2. The hottest [tourist attraction](https://links>

In [159]:
parts = cut_before_and_split(scraped_edition)
print([(i, item) for i, item in enumerate(parts)])

[(0, "[Skip to main content](https://www.morningbrew.com/issues/spacex-ipo-ready-for-blastoff#main-content)\n\nWelcome! We use third-party tools to collect information about you as you enjoy the site. Our [Privacy Policy](https://privacy.morningbrewinc.com/policies?utm_medium=website&utm_source=terms_banner&utm_campaign=mb) explains more. To opt-out of targeted ads, choose [Do Not Sell / Share My Info](https://privacy.morningbrewinc.com/policies?utm_medium=website&utm_source=terms_banner&utm_campaign=mb). By proceeding, you accept our [Terms](https://www.morningbrew.com/terms-service?utm_medium=website&utm_source=terms_banner&utm_campaign=mb) and the [Policy](https://privacy.morningbrewinc.com/policies?utm_medium=website&utm_source=terms_banner&utm_campaign=mb).\n\nGot it\n\n###### ☕SpaceX IPO ready for blastoff\n\nTo:Brew Readers\n\n* * *\n\n|     |     |     |     |\n| --- | --- | --- | --- |\n| |     |\n| --- |\n| Elon Musk's space company made its IPO plans public... |\n\n[![](http

In [162]:
# Filter parts

def filter_parts(scraped_edition):
    split_quiz = scraped_edition.markdown.split("<br>")
    # Define what is being looked for to identify a question or answer and store only those parts
    # seeking = ["1. ", "2. ", "3. ", "4. "]
    q_search = re.compile(r'[1234]\. (“)?.*\[([\w\s\.\-])*\](\(.*\))')
    answer_search = re.compile(r'(\banswer)(s?)(\b)')
    # answers_search = re.compile(r'\banswers\b')
    # hash_answer_search = re.compile(r'(\s### answer)(s)?') I tried this but some editions don't have "###"

    questions_and_answers = [item for item in split_quiz if q_search.search(item.lower()) or (answer_search.search(item.lower()) and item.startswith("[morningbrew.com"))]
    # return [item for item in parts if any(term in item.lower() for term in seeking) or hash_answer_search.search(item.lower())]

    if len(questions_and_answers) > 5:
        del questions_and_answers[5:]

    return questions_and_answers
    

# questions_and_answers = filter_parts(parts)
# questions_and_answers

In [163]:
questions_and_answers = filter_parts(scraped_edition)
questions_and_answers

['🅰️ **Harvard votes to limit A grades.** To crack down on grade inflation, the number of undergraduate A grades that Harvard University instructors can dole out will be [limited](https://www.theguardian.com/education/2026/may/20/harvard-college-cap-a-grade-students?utm_campaign=mb&utm_source=morning_brew&utm_medium=newsletter) to 20% of students in each course, plus an allowance for up to four additional students, beginning in fall 2027. While getting straight A’s will be tougher, there’ll be no limit on A-minus grades. Nearly 70% of Harvard faculty voted to approve the measure, according to results announced yesterday, though it’s overwhelmingly unpopular among students. For context, about two-thirds of letter grades given to Harvard undergraduates during the 2024–25 school year were A’s, up from roughly one-third in 2012–13. The [new cap](https://www.morningbrew.com/stories/harvard-deciding-whether-to-give-fewer-as?utm_campaign=mb&utm_source=morning_brew&utm_medium=newsletter) could

In [135]:
def clean_q4_and_answer(questions_and_answers):

    # Find the answer and extract and store it
    if len(questions_and_answers) == 4:
        answer_rough = questions_and_answers[3].lower().split("answer")[1]
        answer = answer_rough.strip(" |\n")
        questions_and_answers.append(answer)
        
    else:
        answer_index = questions_and_answers[4].lower().find("answer")
        answer_rough = questions_and_answers[4][answer_index:]
        answer_parts = answer_rough.split("|")
        answer_parts_stripped = [item.strip().strip("\n-") for item in answer_parts]
        answer_parts_stripped[0] = ""
        answer_list = []
        [answer_list.append(item) for item in answer_parts_stripped if item]
        answer = answer_list[0]
        questions_and_answers[4] = answer

    # Clean up question 4
    questions_and_answers[3] = (questions_and_answers[3].split("|"))[0]
    return questions_and_answers

# questions_and_answers = clean_q4_and_answer(questions_and_answers)
# questions_and_answers

In [136]:
questions_and_answers = clean_q4_and_answer(questions_and_answers)
questions_and_answers

['1. Tyga’s _$TARFACE_ album [receives](https://links.morningbrew.com/c/U6n?mblid=f92dc3fd9d90&mbnlid=slV5a3vuYndT7PDVh24Aj8) Pitchfork’s first 0/10 score in 19 years',
 '2. The hottest [tourist attraction](https://links.morningbrew.com/c/U6o?mblid=6222a3e47f01&mbnlid=slV5a3vuYndT7PDVh24Aj8) in Tonopah, NV, is this 74-year-old woman’s living room',
 '3. High school football player in Arkansas [practices](https://links.morningbrew.com/c/U6p?mblid=5ac37e49abf9&mbnlid=slV5a3vuYndT7PDVh24Aj8) with a venomous snake in his helmet',
 '4. Beth, the beloved vacuum of a 92-year-old Iowa senator, has [died](https://links.morningbrew.com/c/U6q?mblid=0d6a557bf158&mbnlid=slV5a3vuYndT7PDVh24Aj8) ',
 'we made up the one about the popular living room.']

In [137]:
# Headline text and url extraction

# def morning_brew_link_unpack(url):
#     response = requests.get(url)
#     return soup.find('link', attrs={'rel': 'canonical'}).get('href')
    

def url_headline_extract(index, questions_and_answers):
    url_re = re.compile(r'(\(.*\))')
    url = url_re.search(questions_and_answers[index]).group().strip("()")
    headline = questions_and_answers[index].replace(url, "").replace("()", "").replace("[", "").replace("]", "").lstrip("1234. “").rstrip("”")
    # print(headline)

    if "morningbrew" in url:
        url = urlsplit(requests.get(url, allow_redirects=False, timeout=10).headers.get("Location"))._replace(query="").geturl()

    # if any(term in headline for term in ["“", "”"]):
    #     headline_re = re.compile(r'(“.*”)')
    #     headline = headline_re.search(headline).group().strip("“”")
    # else:

    return headline, url

In [138]:
quiz = {
    "id": id,
    "edition_date": date.isoformat(),
    "source_url": f"https://www.morningbrew.com/issues/{slug}",
    "headlines": [],
    "answer": questions_and_answers[4]
}

for index, item in enumerate(questions_and_answers):
    if item[0].isalpha():
        continue
    else:
        headline, url = url_headline_extract(index, questions_and_answers)
        quiz["headlines"].append({"text": headline, "url": url})

quiz

{'id': 'slV5a3vuYndT7PDVh24Aj8',
 'edition_date': '2026-08-13',
 'source_url': 'https://www.morningbrew.com/issues/what-about-bob',
 'headlines': [{'text': 'Tyga’s _$TARFACE_ album receives Pitchfork’s first 0/10 score in 19 years',
   'url': 'https://www.hotnewhiphop.com/1005560-tyga-starface-pitchfork-review'},
  {'text': 'The hottest tourist attraction in Tonopah, NV, is this 74-year-old woman’s living room',
   'url': 'https://giphy.com/gifs/CBSAllAccess-season-2-episode-7-sabrina-the-teenage-witch-kBH4I91xndN5pTPsIj'},
  {'text': 'High school football player in Arkansas practices with a venomous snake in his helmet',
   'url': 'https://apnews.com/article/venomous-snake-helmet-b4340a60e61757f937eb370b11b80098'},
  {'text': 'Beth, the beloved vacuum of a 92-year-old Iowa senator, has died ',
   'url': 'https://www.nytimes.com/2026/08/08/us/politics/beth-vacuum-chuck-grassley-iowa.html'}],
 'answer': 'we made up the one about the popular living room.'}

In [69]:
print(quiz)

{'id': 'slV5a3vuYndT7PDVh24Aj8', 'issue_date': datetime.date(2026, 8, 13), 'source_url': 'https://www.morningbrew.com/issues/what-about-bob', 'headlines': [{'text': 'Tyga’s _$TARFACE_ album receives Pitchfork’s first 0/10 score in 19 years', 'url': 'https://www.hotnewhiphop.com/1005560-tyga-starface-pitchfork-review'}, {'text': 'The hottest tourist attraction in Tonopah, NV, is this 74-year-old woman’s living room', 'url': 'https://giphy.com/gifs/CBSAllAccess-season-2-episode-7-sabrina-the-teenage-witch-kBH4I91xndN5pTPsIj'}, {'text': 'High school football player in Arkansas practices with a venomous snake in his helmet', 'url': 'https://apnews.com/article/venomous-snake-helmet-b4340a60e61757f937eb370b11b80098'}, {'text': 'Beth, the beloved vacuum of a 92-year-old Iowa senator, has died ', 'url': 'https://www.nytimes.com/2026/08/08/us/politics/beth-vacuum-chuck-grassley-iowa.html'}], 'answer': 'we made up the one about the popular living room.'}
